# Preventing Overfitting with L1 and L2 Penalties

## Introduction

In Lesson 2, we built a linear regression model with one feature — apartment size — and watched it outperform the mean baseline. But a single-feature model has a fundamental limitation: it ignores everything else that drives apartment prices. Neighborhoods, coordinates, building characteristics — all of these signals are left on the table.

When we add all of those features back, a new danger emerges: **overfitting**. With 50+ neighborhood indicator columns alone, the model has enough parameters to memorize the training set rather than learning generalizable patterns. Regularization is the principled solution.

### The Regularization Idea

Regularization modifies the model's objective: instead of minimizing only the prediction error, it minimizes **prediction error plus a penalty for large coefficients**:

$$\text{Loss}_{\text{Regularized}} = \underbrace{\text{Prediction Error}}_{\text{want this low}} + \underbrace{\alpha \times \text{Coefficient Penalty}}_{\text{controls complexity}}$$

The penalty discourages the model from relying too heavily on any single feature. It forces the coefficients to be small unless the evidence from the data is overwhelming — a form of built-in skepticism that prevents overfitting.

**Two penalty functions, two behaviors:**

| Method | Penalty | Effect on coefficients |
|--------|---------|------------------------|
| **Ridge** (L2) | $\alpha \sum \beta_j^2$ | Shrinks all coefficients toward zero — none reach exactly zero |
| **Lasso** (L1) | $\alpha \sum \|\beta_j\|$ | Can shrink coefficients to exactly zero — performs feature selection |

The parameter $\alpha$ controls how strong the penalty is. Small $\alpha$ → model behaves like ordinary linear regression. Large $\alpha$ → model drives all coefficients toward zero.

### What This Lesson Covers

By the end of this lesson, you will be able to:

1. Explain why **feature scaling** is mandatory before applying Ridge or Lasso regularization
2. Build a scikit-learn **Pipeline** that chains encoding, scaling, and modeling — and understand why the pipeline structure prevents data leakage
3. Train **Ridge regression** (L2 penalty) and understand its shrinkage behavior
4. Train **Lasso regression** (L1 penalty) and understand its feature-selection property
5. Tune the regularization strength $\alpha$ and diagnose where a model sits on the **bias-variance** spectrum
6. Use **RidgeCV** and **LassoCV** to select $\alpha$ automatically via cross-validation
7. Extract and interpret model coefficients in the context of standardized features

---

## 1. Data Loading and Splitting

This time, we use all available features — not just apartment size. Our feature matrix includes:

- `surface_covered_in_m2` — apartment size (numeric)
- `lat`, `lon` — geographic coordinates (numeric)
- `neighborhood` — categorical variable with 50+ unique values (will be one-hot encoded)

> **Why does adding `neighborhood` require regularization?**
>
> One-hot encoding 50+ neighborhoods creates 50+ binary columns. Each column is sparse — most apartments are not in Palermo, most are not in Recoleta. With 50+ parameters to learn from sparse columns, the model has ample opportunity to overfit: it can learn "apartments in Palermo block 12 sold for exactly this much" rather than the general Palermo price premium. Regularization controls this by shrinking the neighborhood coefficients, preventing the model from over-learning neighborhood-specific noise.

**Code Task 2.3.1.1**

> 📌 **What changes from Lesson 2 to Lesson 3?**
>
> In Lesson 2, we used only `surface_covered_in_m2` — a single numeric feature — to keep the model simple and visualizable. In this lesson, we add `lat`, `lon`, and the full `neighborhood` column. This expansion:
>
> - **Increases predictive potential:** Location and neighborhood are among the strongest real-estate price predictors. Adding them should reduce both MAE and RMSE substantially.
> - **Increases model complexity:** 50+ neighborhood categories → 50+ binary columns after one-hot encoding → many more coefficients to estimate. Without regularization, this risks overfitting.
> - **Requires a Pipeline:** The one-hot encoder must be fit only on training data, then applied to test data using training-set categories. If a neighborhood appeared only in training but not test (or vice versa), the scaler must handle that gracefully — which a correctly configured Pipeline does automatically.

In [ ]:
# Import clean_files from wrangle.py
from wrangle import clean_files

# Load data
df = clean_files("./data/buenos-aires-real-estate-*.csv")

# Define target and features
target_col = "price_aprox_usd"
feature_cols = [col for col in df.columns if col != target_col]

X = df[feature_cols]
y = df[target_col]

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Feature columns: {X.columns.tolist()}")

### 1.1 Train-Test Split

We use the same `random_state=42` and `test_size=0.2` as Lessons 1 and 2. This is not just convention — it is a requirement for fair model comparison. In Lesson 4, we will place all models (linear, Ridge, Lasso) side by side in a single performance table. That comparison is only valid if every model trained and evaluated on identical data. Any difference in the split would confound the results: we would not know whether Ridge outperforms linear regression because it is a better model or simply because it happened to get easier test examples.

Note that the suffix `_3` in `X_train_3`, `X_test_3`, `y_train_3`, `y_test_3` distinguishes these variables from Lessons 1 and 2 in case you are running all lessons in the same Jupyter environment.

**Code Task 2.3.1.2**

In [ ]:
from sklearn.model_selection import train_test_split

# Perform train-test split
X_train_3, X_test_3, y_train_3, y_test_3 = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set: {X_train_3.shape[0]} samples")
print(f"Test set: {X_test_3.shape[0]} samples")

In [ ]:
from IPython.display import VimeoVideo

# Bigger video
VimeoVideo("1169844027", h="3298dbabb7", width=700, height=450) 

## Setup: Import Libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import Ridge, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score
from category_encoders import OneHotEncoder

**Now it is time to answer MCQ 2.3.1.1.**

## 2. The Need for Feature Scaling

Before we can apply regularization, we must address a mathematical prerequisite: **feature scaling**.

### Why scaling is mandatory for regularized models

Regularization penalizes large coefficients. But "large" depends entirely on the units of the feature. A coefficient of `1000` on `surface_covered_in_m2` (measured in m²) represents the same price-per-m² relationship as a coefficient of `0.01` on a feature measured in cm² — same real-world effect, wildly different numeric values. Without scaling, the regularization penalty treats these unequally: it aggressively shrinks the 1000 coefficient while barely touching the 0.01 coefficient, even though they encode equivalent information.

> **The squaring problem (Ridge / L2)**
>
> Ridge's penalty squares each coefficient: $\alpha \sum \beta_j^2$.
>
> **Unscaled example:**
> - Feature A (`surface_covered_in_m2`): coefficient = 1,000 → penalty = 1,000² = **1,000,000**
> - Feature B (`lat`): coefficient = 100,000 → penalty = 100,000² = **10,000,000,000**
>
> Feature B is penalized 10,000× more than Feature A despite potentially contributing equivalent information. Ridge will unfairly suppress Feature B.
>
> **After StandardScaler (mean=0, std=1):**
> - Feature A (scaled): coefficient ≈ 50 → penalty = 50² = 2,500
> - Feature B (scaled): coefficient ≈ 50 → penalty = 50² = 2,500
>
> Both features are on equal footing. The penalty is applied fairly based on statistical importance, not on the accident of measurement units.

**StandardScaler** applies the transformation:

$$z = \frac{x - \mu}{\sigma}$$

where $\mu$ is the column mean and $\sigma$ is the standard deviation, both computed from the training data. After transformation, every feature has mean = 0 and standard deviation = 1.

> ⚠️ **Critical:** StandardScaler must be fit **only on training data** and then applied to test data using the training statistics. If we fit the scaler on the combined dataset (train + test), test-set statistics influence the training transformation — that is data leakage. This is exactly why we build models inside a **Pipeline**: the pipeline automatically applies transformers only where they should be applied.

**Now it is time to answer MCQ 2.3.2.1.** & **MCQ 2.3.2.2.**

---

## 3. Ridge Regression Pipeline

Now we build our first regularized model. Instead of applying encoding and scaling manually, we will use a scikit-learn **Pipeline** that bundles all preprocessing and modeling steps into a single object.

### Why sklearn Pipeline — and why it prevents data leakage

The pipeline is not merely a convenience. It is the architecturally correct way to build a model that preprocesses data before training.

**The data leakage risk without a Pipeline:**

```python
# WRONG — data leakage!
ohe = OneHotEncoder(use_cat_names=True)
scaler = StandardScaler()

# These fit on the entire dataset — including test data
X_encoded = ohe.fit_transform(X)
X_scaled = scaler.fit_transform(X_encoded)

# Split AFTER fitting transformers — test statistics already contaminated training
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y)
```

When the encoder or scaler is fit on the full dataset (train + test), it learns statistics (category frequencies, means, standard deviations) from the test set. The model thus has indirect access to test-set information before it is supposed to be "hidden" — producing unrealistically good test-set performance. This is data leakage.

**The Pipeline solution:**

```python
# CORRECT — no data leakage
model = make_pipeline(
    OneHotEncoder(use_cat_names=True),   # Step 1: encode categories
    StandardScaler(),                     # Step 2: scale to mean=0, std=1
    Ridge(alpha=1.0)                      # Step 3: fit regularized model
)

model.fit(X_train, y_train)   # All transformers fit ONLY on X_train
model.predict(X_test)          # Transformers applied to X_test using training statistics
```

When `model.fit(X_train, y_train)` is called:
1. `OneHotEncoder` is fit on `X_train` only — learns which categories exist in training data
2. `StandardScaler` is fit on the encoded `X_train` only — learns mean and std from training data
3. `Ridge` is fit on the scaled, encoded `X_train` with `y_train` as labels

When `model.predict(X_test)` is called:
1. `OneHotEncoder` transforms `X_test` using categories learned from training
2. `StandardScaler` scales `X_test` using mean/std learned from training
3. `Ridge` predicts using the fitted coefficients

The test set is always transformed using training-data statistics. No test-set information ever influences any fitted parameters.

> **Why `category_encoders` over scikit-learn's `OneHotEncoder`?**
>
> Scikit-learn's `OneHotEncoder` returns anonymous column names: `x0_0`, `x0_1`, `x0_2`. After encoding 50+ neighborhoods, you cannot tell which column corresponds to which neighborhood without consulting a separate lookup table.
>
> `category_encoders.OneHotEncoder` with `use_cat_names=True` returns descriptive names: `neighborhood_Palermo`, `neighborhood_Recoleta`, `neighborhood_Caballito`. Every coefficient maps directly to a readable neighborhood name. This interpretability is essential when we visualize feature importance in Section 6.

**Code Task 2.3.3.1**

> 📌 **`make_pipeline` vs. `Pipeline`**: Scikit-learn offers two syntaxes for creating pipelines. `make_pipeline(step1, step2, step3)` automatically assigns step names based on class names (e.g., `onehotencoder`, `standardscaler`, `ridge`). `Pipeline([("encode", step1), ("scale", step2), ("model", step3)])` lets you choose the step names explicitly. Either works; `make_pipeline` is more concise for quick experiments, while named `Pipeline` is clearer in production code where you need to access specific steps by name — which we will do in Section 6 when extracting coefficients.

In [ ]:
model_ridge_3 = make_pipeline(
    OneHotEncoder(use_cat_names=True),
    StandardScaler(),
    Ridge(alpha=1.0)
)

# Display the pipeline
model_ridge_3

**Now it is time to answer MCQ 2.3.3.1.**

Now let's fit the pipeline and evaluate it. The `.fit()` and `.predict()` calls work exactly as before — the pipeline wraps all steps transparently.

**Code Task 2.3.3.2**

In [ ]:
# Fit model
model_ridge_3.fit(X_train_3, y_train_3)

# Predict on training data
y_pred_ridge_train = model_ridge_3.predict(X_train_3)
rmse_ridge_train = root_mean_squared_error(y_train_3, y_pred_ridge_train)

# Predict on test data
y_pred_ridge_test = model_ridge_3.predict(X_test_3)
rmse_ridge_test = root_mean_squared_error(y_test_3, y_pred_ridge_test)

print(f"Ridge Training RMSE: ${rmse_ridge_train:,.2f}")
print(f"Ridge Test RMSE: ${rmse_ridge_test:,.2f}")

**Now it is time to answer MCQ 2.3.3.2.**

---

## 4. Lasso Regression and Feature Selection

Lasso (Least Absolute Shrinkage and Selection Operator) applies a different penalty function than Ridge — the **L1 penalty** — and this seemingly small mathematical change produces a dramatically different behavior: Lasso can reduce coefficients to exactly zero, effectively removing features from the model.

### Why the L1 penalty creates sparsity

**Ridge (L2) penalty:** $\alpha \sum \beta_j^2$

The squared penalty makes the cost of reducing a coefficient from 0.001 to 0 almost negligible (0.001² is tiny). This means Ridge never finds it optimal to drive a coefficient to exactly zero — it just makes them very small.

**Lasso (L1) penalty:** $\alpha \sum |\beta_j|$

The absolute-value penalty costs the same to reduce any coefficient by the same amount, regardless of the coefficient's current value. This creates a mathematical incentive to eliminate marginal features entirely — driving their coefficients precisely to zero is often optimal under the L1 penalty.

**The practical consequence:** Lasso performs **automatic feature selection**. Features with insufficient predictive power get zeroed out. The model that emerges is **sparse** — most coefficients are zero, and only the truly important features retain non-zero coefficients.

### Ridge vs. Lasso: A Full Comparison

| Property | Ridge (L2) | Lasso (L1) |
|----------|-----------|-----------|
| Penalty term | $\alpha \sum \beta_j^2$ | $\alpha \sum \|\beta_j\|$ |
| Can set coefficients to exactly zero? | No — coefficients approach zero but never reach it | Yes — coefficients can be exactly zero |
| Feature selection? | No | Yes — automatic |
| Preferred when... | All features contribute something; you want to shrink all of them | Many features are noise; you want the model to identify the important ones |
| Interpretability | Coefficients for all features; harder to identify the important ones | Sparse set of non-zero coefficients; easy to identify important features |
| Numerical stability | Very stable (closed-form solution) | Less stable for correlated features (which one gets zeroed?) |

> 🧠 **The geometry behind the difference:**
>
> The L2 penalty creates a circular constraint region — the OLS solution is pulled toward the center (all zeros) from any direction, always landing somewhere with non-zero components.
>
> The L1 penalty creates a diamond-shaped constraint region — the corners of the diamond are on the coordinate axes, so the OLS solution frequently lands at a corner (where one or more coefficients are exactly zero).

**Code Task 2.3.4.1**

In [ ]:
model_lasso_3 = make_pipeline(
    OneHotEncoder(use_cat_names=True),
    StandardScaler(),
    Lasso(alpha=100, max_iter=10000)
)

model_lasso_3.fit(X_train_3, y_train_3)

# Calculate test RMSE
y_pred_lasso_test = model_lasso_3.predict(X_test_3)
rmse_lasso_test = root_mean_squared_error(y_test_3, y_pred_lasso_test)

print(f"Lasso Test RMSE: ${rmse_lasso_test:,.2f}")

**Now it is time to answer MCQ 2.3.4.1.**

---

## 5. Hyperparameter Tuning: Finding the Right Alpha

The regularization strength $\alpha$ is a **hyperparameter** — a setting you choose before training, not one the model learns from data. Getting it right is critical.

### The bias-variance tradeoff

$\alpha$ controls a fundamental tradeoff between two types of error:

**Bias** is the error from using too simple a model. A model with very high bias systematically misses real patterns in the data — it is not flexible enough to fit the training set well, let alone generalize.

**Variance** is the error from using too complex a model. A high-variance model fits training data so closely that small changes in the training set would produce wildly different predictions — it has memorized noise instead of learning signal.

> **The $\alpha$ dial:**
>
> - **$\alpha \to 0$:** No regularization — model is ordinary linear regression. Risk of high variance (overfitting): the model can fit training data perfectly but may not generalize.
> - **$\alpha \to \infty$:** Maximum regularization — all coefficients driven to zero. Risk of high bias (underfitting): the model makes the same prediction for every apartment regardless of features.
> - **Optimal $\alpha$:** Somewhere between these extremes — model is complex enough to capture real patterns but constrained enough to avoid memorizing noise.

This tradeoff produces a characteristic **U-shaped performance curve** when test error is plotted against $\alpha$:
- Left of the U: low $\alpha$ → overfitting → large train-test gap
- Bottom of the U: optimal $\alpha$ → best generalization
- Right of the U: high $\alpha$ → underfitting → both train and test error rise

How much of this shape you actually see depends on the dataset. When there are many more rows than features, the left arm is flat rather than steep: the unregularized model is not overfitting, so there is no variance for $\alpha$ to remove. You will see exactly that below.

### 5.1 Ridge: Exploring the Bias-Variance Tradeoff

**Code Task 2.3.5.1**: Calculate training and test RMSE for Ridge across seven alphas `[0.001, 0.01, 0.1, 1, 10, 100, 1000]`.

In [ ]:
# Test seven alpha values across 6 orders of magnitude
alphas = [0.001, 0.01, 0.1, 1, 10, 100, 1000]
ridge_results = []

for a in alphas:
    # Create and fit pipeline
    model = make_pipeline(
        OneHotEncoder(use_cat_names=True),
        StandardScaler(),
        Ridge(alpha=a)
    )
    model.fit(X_train_3, y_train_3) # <-- fit on train

    # Calculate RMSE for training and test
    rmse_train = root_mean_squared_error(y_train_3, model.predict(X_train_3))
    rmse_test = root_mean_squared_error(y_test_3, model.predict(X_test_3))

    ridge_results.append({
        "alpha": a,
        "rmse_train": rmse_train,
        "rmse_test": rmse_test
    })

df_ridge_results = pd.DataFrame(ridge_results)
print(df_ridge_results)

**Code 2.3.5.2**: Create a line plot showing the bias-variance tradeoff for Ridge. Plot training RMSE and test RMSE on the y-axis, with alpha on the x-axis (log scale).

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
ax.plot(df_ridge_results["alpha"], df_ridge_results["rmse_train"],
        marker='o', linewidth=2, markersize=8, label="Training RMSE (Low Bias)")
ax.plot(df_ridge_results["alpha"], df_ridge_results["rmse_test"],
        marker='s', linewidth=2, markersize=8, label="Test RMSE (Generalization)")
ax.axvline(x=1, color='red', linestyle='--', alpha=0.5, label="α=1 (Our choice)")

# Where each curve must converge as alpha -> infinity (model predicts the training mean)
ax.axhline(y_train_3.std(), color='C0', linestyle=':', alpha=0.7, label="sd(y_train): α→∞ limit")
ax.axhline(y_test_3.std(), color='C1', linestyle=':', alpha=0.7, label="sd(y_test): α→∞ limit")
ax.set_xlabel("alpha (Regularization Strength)", fontsize=12)
ax.set_ylabel("RMSE ($)", fontsize=12)
ax.set_xscale("log")
ax.set_title("Ridge Regression: The Bias-Variance Tradeoff", fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

> **Interpreting the Curve**
>
> The plot shows the bias-variance tradeoff, but not as the symmetric U of the textbook diagram. Read it carefully:
>
> - **Left side (low $\alpha$):** both curves are flat. After one-hot encoding we have roughly 53 features and several thousand training rows. With $n \gg p$ the unregularized model is not overfitting, so shrinking coefficients buys nothing and costs nothing. The steep left arm of the classic U only appears when $p$ is comparable to $n$.
>
> - **Where the curve turns ($\alpha \approx 10^3$):** because the features are standardized, $\alpha$ only starts to bite once it is comparable in size to the eigenvalues of $X^\top X$, which grow with the number of training rows. Below that threshold the penalty is negligible next to the fit term, which is why nothing happens over the first six orders of magnitude.
>
> - **Right side (high $\alpha$):** both errors rise and then flatten. The model collapses toward predicting the training mean for every apartment, so training RMSE converges to $\text{sd}(y_{\text{train}})$ and test RMSE converges to $\text{sd}(y_{\text{test}})$ adjusted for the gap between the two means. **These are two different constants: the curves flatten out, but they do not converge to each other.** The dotted reference lines mark exactly these two limits.
>
> - **Why can training RMSE sit above test RMSE?** Apartment prices are right-skewed and RMSE squares the errors, so a handful of extreme apartments dominates the number. A single 80/20 split can leave fewer of those extremes in the test set, which pushes test RMSE below training RMSE. This is a property of the split, not a defect in the model, and it is one more reason to prefer cross-validation over a single split.
>
> **The diagnostic rule:** If train RMSE ≪ test RMSE → increase $\alpha$ (reduce variance). If both sit near their $\alpha \to \infty$ limits → decrease $\alpha$ (reduce bias). What this particular curve tells you is that the dataset sits firmly in the low-variance regime: there is no overfitting left to correct, so the only thing $\alpha$ can do here is add bias.

**Now it is time to answer MCQ 2.3.5.1.**

### 5.2 Lasso: Feature Selection with Regularization

Lasso behaves similarly to Ridge at extreme $\alpha$ values, but the middle of the U-curve has a different character: at intermediate $\alpha$ values, Lasso is simultaneously reducing overfitting *and* selecting features. The number of non-zero coefficients decreases monotonically as $\alpha$ increases.

**Code Task 2.3.5.3**: Create a Lasso tuning loop testing alphas `[0.1, 1, 10, 100, 1000]`.

In [ ]:
lasso_results = []
alphas_lasso = [0.1, 1, 10, 100, 1000]

for a in alphas_lasso:
    model = make_pipeline(
        OneHotEncoder(use_cat_names=True),
        StandardScaler(),
        Lasso(alpha=a, max_iter=10000)
    )
    model.fit(X_train_3, y_train_3)  # <-- fit on train

    rmse_train = root_mean_squared_error(y_train_3, model.predict(X_train_3))
    rmse_test = root_mean_squared_error(y_test_3, model.predict(X_test_3))
    non_zero_features = (model.named_steps["lasso"].coef_ != 0).sum()

    lasso_results.append({
        "alpha": a,
        "rmse_train": rmse_train,
        "rmse_test": rmse_test,
        "non_zero_features": non_zero_features
    })

df_lasso_results = pd.DataFrame(lasso_results)
print(df_lasso_results)

**Code 2.3.5.4**: Compare Ridge and Lasso alpha selection visually.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 6))

# Ridge comparison
ax1.plot(df_ridge_results["alpha"], df_ridge_results["rmse_test"],
         marker='o', linewidth=2, markersize=8, label="Ridge Test RMSE")
ax1.axvline(x=1, color='red', linestyle='--', alpha=0.7, linewidth=2, label="α=1 (Ridge default)")
ax1.set_xlabel("alpha (Regularization Strength)")
ax1.set_ylabel("Test RMSE ($)")
ax1.set_xscale("log")
ax1.set_title("Ridge: Alpha vs Test RMSE")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Lasso comparison with feature count
ax2_twin = ax2.twinx()
ax2.plot(df_lasso_results["alpha"], df_lasso_results["rmse_test"],
         marker='s', linewidth=2, markersize=8, color='green', label="Lasso Test RMSE")
ax2_twin.plot(df_lasso_results["alpha"], df_lasso_results["non_zero_features"],
              marker='^', linewidth=2, markersize=8, color='orange', linestyle='--', label="Non-zero Features")
ax2.axvline(x=100, color='red', linestyle='--', alpha=0.7, linewidth=2, label="α=100 (Our choice)")
ax2.set_xlabel("alpha (Regularization Strength)")
ax2.set_ylabel("Test RMSE ($)", color='green')
ax2_twin.set_ylabel("Number of Non-zero Features", color='orange')
ax2.set_xscale("log")
ax2.set_title("Lasso: Alpha vs Test RMSE & Feature Count")
ax2.legend(loc='upper left')
ax2_twin.legend(loc='upper right')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

> **Why $\alpha = 100$ for Lasso?**
>
> From the tuning loop results, you can see distinct behavior across the range:
>
> - **Low $\alpha$ (0.1):** Lasso keeps most features active. Performance is good but the model may be retaining noise features. The model is closer to ordinary linear regression.
> - **$\alpha = 100$:** A good balance — aggressive enough to eliminate many noise features (sparse model) while keeping test RMSE reasonable. This is the "feature selection regime" where Lasso's unique strength is most visible.
> - **High $\alpha$ (1000):** Lasso has driven so many coefficients to zero that even useful features are being eliminated. Test RMSE rises — underfitting.
>
> $\alpha = 100$ is in the regime where Lasso performs meaningful feature selection without sacrificing too much predictive accuracy. However, the optimal $\alpha$ depends on your specific dataset — which is precisely why we use **LassoCV** to find it automatically.

### 5.3 Automated Alpha Selection: RidgeCV and LassoCV

Manually testing a grid of $\alpha$ values and eyeballing the U-curve is informative but not rigorous. **Cross-validation** provides a principled, statistically reliable method for selecting $\alpha$.

> **What is cross-validation?**
>
> Instead of a single train-test split, cross-validation divides the training data into $k$ folds (typically 5 or 10). For each candidate $\alpha$:
>
> 1. Train on $k-1$ folds, evaluate on the held-out fold
> 2. Repeat $k$ times (each fold takes a turn as the held-out set)
> 3. Average the $k$ error estimates
> 4. Select the $\alpha$ with the lowest average cross-validation error
>
> This approach uses all available training data for both training and validation, producing a more reliable estimate of generalization performance than a single split. A single train-test split estimate of performance has high variance — two different splits of the same data can yield quite different "optimal" alphas. Cross-validation averages out this variance by using all possible held-out folds.

`RidgeCV` and `LassoCV` implement this procedure automatically — you simply pass a list of candidate alphas and the model selects the best one internally. The selected alpha is stored in `model.named_steps["ridgecv"].alpha_` after fitting.

**Code 2.3.5.5**

In [ ]:
from sklearn.linear_model import RidgeCV, LassoCV

# RidgeCV automatically finds the best alpha
ridge_cv_model = make_pipeline(
    OneHotEncoder(use_cat_names=True),
    StandardScaler(),
    RidgeCV(alphas=[0.001, 0.01, 0.1, 1, 10, 100, 1000, 10000, 100000, 1000000], cv=5)  # <--- 5-fold cross-validation
)
ridge_cv_model.fit(X_train_3, y_train_3)

# Extract the optimal alpha
optimal_alpha_ridge = ridge_cv_model.named_steps["ridgecv"].alpha_
print(f"RidgeCV selected alpha: {optimal_alpha_ridge}")
print(f"Ridge CV Test RMSE: ${root_mean_squared_error(y_test_3, ridge_cv_model.predict(X_test_3)):,.2f}")

# LassoCV automatically finds the best alpha
lasso_cv_model = make_pipeline(
    OneHotEncoder(use_cat_names=True),
    StandardScaler(),
    LassoCV(alphas=[0.1, 1, 10, 100, 1000], cv=5, max_iter=10000)
)
lasso_cv_model.fit(X_train_3, y_train_3)

optimal_alpha_lasso = lasso_cv_model.named_steps["lassocv"].alpha_
print(f"\nLassoCV selected alpha: {optimal_alpha_lasso}")
print(f"Lasso CV Test RMSE: ${root_mean_squared_error(y_test_3, lasso_cv_model.predict(X_test_3)):,.2f}")
print(f"Non-zero features (Lasso CV): {(lasso_cv_model.named_steps['lassocv'].coef_ != 0).sum()}")

> **RidgeCV and LassoCV — Key Properties**
>
> - `RidgeCV(alphas=[...])` automatically selects the best $\alpha$ from the provided list using cross-validation. The chosen alpha is stored in `model.alpha_` after fitting.
> - `LassoCV(cv=5)` similarly selects the optimal alpha and stores it in `model.alpha_`. The `cv` parameter specifies the number of folds.
> - Both integrate naturally into Pipelines: `make_pipeline(OHE, Scaler, RidgeCV(alphas=[...]))` works exactly like the non-CV versions.
>
> **Why not always use CV?** Cross-validation is computationally more expensive — it fits the model $k \times |\text{alphas}|$ times instead of once. For small datasets and fast models (like Ridge and Lasso), this cost is negligible. For large datasets or slow models, you might accept a slightly suboptimal $\alpha$ to save computation.

---

## 6. Visualizing and Interpreting Coefficients

The true power of Lasso's feature selection becomes visible when we examine the coefficients directly. Let's extract them and see which features the model kept and which it zeroed out.

**Code Task 2.3.6.1**: Extract the coefficients and feature names from `model_lasso_3`. Create a Series named `feat_imp_3`.

In [ ]:
# Extract coefficients from the Lasso step
coefficients = model_lasso_3.named_steps["lasso"].coef_

# Extract feature names from the OneHotEncoder step
features = model_lasso_3.named_steps["onehotencoder"].get_feature_names_out()

# Create Series
feat_imp_3 = pd.Series(coefficients, index=features)

print(f"Total features: {len(feat_imp_3)}")
print(f"Non-zero features: {(feat_imp_3 != 0).sum()}")

> **Sparsity as a Competitive Advantage**
>
> Observe the dramatic feature compression Lasso achieved:
>
> - **Before encoding:** ~5 raw columns (surface area, lat, lon, neighborhood as one column)
> - **After OneHotEncoder:** ~53 features (neighborhood alone expands to 50+ binary columns)
> - **After Lasso ($\alpha = 100$):** ~20 features with non-zero coefficients — roughly 62% of features zeroed out
>
> This is **automatic feature selection** in action. Lasso identified which of the 50+ neighborhood columns genuinely predict price and which are noise, eliminating the noise without human intervention.
>
> Why does this matter?
>
> 1. **Interpretability:** A model with 20 non-zero features is far easier to explain than one with 53. You can enumerate which neighborhoods matter and quantify their price effect.
> 2. **Computational efficiency:** Fewer active features mean faster predictions in production.
> 3. **Generalization:** By eliminating features with insufficient signal, Lasso reduces the chance of overfitting to training-set peculiarities.
>
> This feature-selection property is Lasso's defining advantage over Ridge for high-dimensional problems where many features are suspected to be noise.

Now let's visualize the top coefficients.

**Code 2.3.6.2**: Plot the top 10 features by absolute coefficient value.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
feat_imp_3.sort_values(key=abs).tail(10).plot(kind="barh", ax=ax)
ax.set_title("Top 10 Coefficients: Lasso (alpha=100)")
ax.set_xlabel("Coefficient Value")
ax.set_ylabel("Feature")
plt.tight_layout()
plt.show()

**Now it is time to answer MCQ 2.3.6.1.**

**Code Task 2.3.6.3**: Count how many features have been set to exactly zero by the Lasso model.

In [ ]:
lasso_zeros_count = (feat_imp_3 == 0).sum()
print(f"Lasso removed {lasso_zeros_count} features out of {len(feat_imp_3)} total features.")
print(f"Percentage removed: {lasso_zeros_count / len(feat_imp_3) * 100:.1f}%")

**Verify 2.3.6.3**

In [ ]:
assert 'lasso_zeros_count' in globals(), "❌ 'lasso_zeros_count' is not defined"
assert isinstance(lasso_zeros_count, (int, np.integer)), "❌ Should be an integer count"
assert lasso_zeros_count >= 0, "❌ Zero count should be non-negative"
assert lasso_zeros_count < len(feat_imp_3), "❌ Should not eliminate all features"
assert lasso_zeros_count + (feat_imp_3 != 0).sum() == len(feat_imp_3), "❌ Zero and non-zero counts should sum to total"
print("✓ All assertions passed")

**Now it is time to answer MCQ 2.3.6.2.**

### 6.4 Interpreting Lasso Coefficients

Lasso coefficients require more careful interpretation than OLS coefficients. Two important caveats apply:

**Caveat 1: Coefficients are on the standardized scale**

After `StandardScaler`, every feature has mean = 0 and standard deviation = 1. A coefficient of +50,000 on `neighborhood_Palermo` does not mean "Palermo adds $50,000 to the price in raw square-meter terms." It means Palermo adds $50,000 per *standard deviation* of the scaled feature — which, for a binary indicator column, essentially means $50,000 per unit of being in Palermo.

For binary indicator columns (all our neighborhood columns), the standardized and raw interpretations coincide closely: a coefficient of +$X means being in that neighborhood is associated with approximately +$X relative to the baseline neighborhood, after accounting for all other features.

**Caveat 2: Coefficients reflect the post-selection model**

Lasso has already eliminated ~60% of features. The surviving coefficients are not just estimates of those features' effects in isolation — they are estimates *after the other eliminated features have been removed*. If Lasso zeroed out `neighborhood_Recoleta` but kept `neighborhood_Palermo`, the Palermo coefficient absorbs some of the effect that Recoleta might have partially shared.

> **Practical interpretation guideline:**
>
> For business communication, the cleanest interpretation is relative: "Apartments in neighborhood X are predicted to sell for approximately $Y more (or less) than apartments in the baseline neighborhood, holding size and location fixed." The exact dollar amount is an approximation, but the direction and rough magnitude are reliable.

**Code 2.3.6.4**: Display the top 5 positive and negative features from the fitted Lasso model.

In [ ]:
# Show top 5 positive and negative coefficients
print("=" * 60)
print("TOP 5 POSITIVE FEATURES (Increase Price)")
print("=" * 60)
top_positive = feat_imp_3.nlargest(5)
for feature, coef in top_positive.items():
    print(f"{feature:40} : ${coef:>10,.2f}")

print("\n" + "=" * 60)
print("TOP 5 NEGATIVE FEATURES (Decrease Price)")
print("=" * 60)
top_negative = feat_imp_3.nsmallest(5)
for feature, coef in top_negative.items():
    print(f"{feature:40} : ${coef:>10,.2f}")

print("\n" + "=" * 60)
print("INTERPRETATION GUIDE")
print("=" * 60)
print("Remember: These coefficients are on STANDARDIZED features.")
print("A feature's importance depends on both its coefficient AND")
print("which other features Lasso eliminated. This is the 'feature selection'")
print("effect of Lasso in action.")

### 6.5 Residual Analysis: Univariate vs. Multivariate

Adding neighborhood, latitude, and longitude to the model should capture more of the systematic price variation that size alone could not explain. Let's verify this by comparing residual patterns.

**Hypothesis:** The multivariate residuals should be more randomly scattered than the univariate residuals from Lesson 2, because the model now captures neighborhood-level price effects that were previously "missing" and showing up as systematic patterns.

**Reading the residual plot for a multivariate model:**

With a single feature, we could plot residuals against that feature and directly see the pattern. With multiple features (surface area + lat + lon + 50+ neighborhood indicators), we plot residuals against the **predicted values** instead — the only single axis that summarizes all feature contributions.

The same interpretive rules apply:
- **Random scatter around zero:** All systematic variation is captured; remaining errors are unpredictable noise.
- **Funnel shape (heteroscedasticity):** Error variance increases with predicted price — a common pattern in financial data that linear models struggle to fully resolve.
- **Systematic curve:** The relationship is non-linear in ways the model's current features cannot capture.

**Code 2.3.6.5**: Create a residual plot for the multivariate Lasso model and compare it to the univariate model's residuals.

In [ ]:
# Get residuals from Lasso predictions on test set
y_pred_lasso = model_lasso_3.predict(X_test_3)
residuals_lasso = y_test_3 - y_pred_lasso

# Create residual plot
fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(y_pred_lasso, residuals_lasso, alpha=0.5, s=30)
ax.axhline(y=0, color='r', linestyle='--', linewidth=2, label='Zero Error')
ax.set_xlabel("Predicted Price (USD)", fontsize=12)
ax.set_ylabel("Residuals (USD)", fontsize=12)
ax.set_title("Lasso (Multivariate) Model: Residual Plot", fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Summary statistics
print(f"Mean of residuals: ${residuals_lasso.mean():,.2f} (should be close to 0)")
print(f"Std of residuals: ${residuals_lasso.std():,.2f}")
print(f"Min residual: ${residuals_lasso.min():,.2f}")
print(f"Max residual: ${residuals_lasso.max():,.2f}")

> **Comparing Univariate (Lesson 2) vs. Multivariate (Lesson 3) Residuals**
>
> **Univariate model (Lesson 2) — one feature: surface area**
> - Residuals showed systematic funnel-shaped heteroscedasticity — variance increased at higher prices
> - Systematic underestimation for some neighborhoods, overestimation for others
> - The model could not distinguish a 50m² apartment in Palermo from a 50m² apartment in a peripheral neighborhood
>
> **Multivariate model (Lesson 3) — surface area + coordinates + neighborhood**
> - Residuals should be more randomly scattered — less systematic structure
> - The model now differentiates apartments by location, not just size
> - The funnel shape may still be present (heteroscedasticity is a real property of price data) but should be less severe
>
> ❗️ **If the multivariate residuals still show systematic patterns**, it means important relationships remain uncaptured — non-linearities in the size-price relationship, interactions between neighborhood and size, or other factors our feature set does not include. Residual analysis is a diagnostic tool: it tells you what your current model is missing.

**Code 2.3.7.1**: Build a Ridge vs. Lasso comparison table showing training and test RMSE for both models.

In [ ]:
ridge_metrics = {
    "Model": "Ridge (α=1.0)",
    "RMSE_train": rmse_ridge_train,
    "RMSE_test": rmse_ridge_test,
    "Non-zero_features": "All"
}

lasso_metrics = {
    "Model": "Lasso (α=100)",
    "RMSE_train": root_mean_squared_error(y_train_3, model_lasso_3.predict(X_train_3)),
    "RMSE_test": rmse_lasso_test,
    "Non-zero_features": (feat_imp_3 != 0).sum()
}

df_comparison = pd.DataFrame([ridge_metrics, lasso_metrics])
print(df_comparison)

---

## Summary

In this lesson, you moved from a single-feature linear model to regularized multi-feature models that can handle high-dimensional data without overfitting. Here is what each step built on the previous:

| Step | What we did | Why it matters |
|------|-------------|----------------|
| **Feature setup** | Added neighborhood (50+ categories), lat, lon to X | More signals → better potential accuracy, but also more overfitting risk |
| **StandardScaler** | Transformed all features to mean=0, std=1 | Ensures the regularization penalty treats all features fairly |
| **Pipeline** | Chained OHE → Scaler → Model in one object | Prevents data leakage from transformers fitted on test data |
| **Ridge (L2)** | Penalty $\alpha \sum \beta_j^2$ — shrinks all coefficients | Reduces variance without removing any features |
| **Lasso (L1)** | Penalty $\alpha \sum \|\beta_j\|$ — zeroes out weak features | Automatic feature selection; sparse, interpretable model |
| **Alpha tuning** | Tested range of alphas; plotted train and test error curves | Found the bias-variance tradeoff visually |
| **RidgeCV / LassoCV** | Cross-validation for automatic alpha selection | Principled, reproducible hyperparameter selection |
| **Coefficient analysis** | Extracted non-zero features from Lasso | Identified which neighborhoods drive price most strongly |
| **Residual comparison** | Compared L2 (univariate) vs L3 (multivariate) residuals | Verified that adding features reduced systematic errors |

**Key Takeaways:**

- **Feature scaling is mandatory for regularization** — without it, the penalty unfairly targets features with different units or ranges
- **Pipelines are architecturally correct** — they are not just convenience; they are the only way to guarantee that transformers are fit only on training data
- **Ridge keeps all features; Lasso selects features** — choose based on whether you believe most features contribute (Ridge) or only some (Lasso)
- **Alpha is the complexity dial** — low alpha ≈ ordinary regression (high variance risk), high alpha ≈ constant prediction (high bias risk), optimal alpha is where the test error curve is at its minimum
- **Lasso coefficients are sparse** — the zero coefficients are the model's statement that those features are not worth including at this alpha level
- **Coefficient interpretation requires care** — Lasso coefficients are on the standardized scale and reflect the post-selection model

**Now it is time to answer MCQ 2.3.7.1.**

---

## Discussion Questions

1. **Ridge vs. Lasso:** Given that our dataset has 50+ neighborhood columns and we believe only a subset of neighborhoods significantly affect price, which regularization method is more appropriate? Why?

2. **Pipeline order:** We applied `StandardScaler` *after* `OneHotEncoder`. Why does this order make sense? What would go wrong if we scaled before encoding?

3. **Alpha tuning:** If your model's training RMSE is much lower than its test RMSE, which direction should you move $\alpha$? Explain why in terms of the bias-variance tradeoff.

4. **Coefficient interpretation:** Look at the top 10 features from your Lasso model. Which neighborhoods have the strongest positive and negative price associations? Do these make intuitive sense given what you know about Buenos Aires?

5. **Residuals:** After adding neighborhood and location features, did the residual plots improve compared to the single-feature model? If systematic patterns remain, what additional features might help?